# 🌍 Spatial-LLM — Training on Kaggle (Qwen2.5-1.5B)

**Kaggle advantages over Colab:** 30 GPU hrs/week, 9-hour sessions, background execution (training continues if you close the tab), reliable T4/P100.

**Setup:** Right sidebar → Settings → Accelerator → **GPU T4 x2** (or P100) → and turn ON **Internet** (required to download the model).

Uses Qwen2.5-1.5B (Apache-2.0, no login needed) — trains fast on a T4.

## 1. Setup

In [ ]:
!git clone https://github.com/Mohammadzamanid/Spatial-LLM.git
%cd Spatial-LLM
!pip install -q transformers peft accelerate datasets bitsandbytes
!pip install -q geopandas shapely mercantile pyyaml pillow
import torch
print('GPU:', torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'CPU — enable GPU in Settings!')
!git log --oneline -1

## 2. Real data (GeoNames world cities)

In [ ]:
!python -m src.data.real_datasets --dataset cities15000 --n_train 8000 --n_val 1000
import json
with open('data/processed/train.jsonl') as f:
    for _, l in zip(range(3), f):
        r = json.loads(l); print('Q:', r['question']); print('A:', r['answer']); print()

## 3. Ablation — which neuro modules contribute (optional, fast)

In [ ]:
!python -m src.eval.ablation --mode leave_one_out

## 4. Train (Qwen2.5-1.5B + spatial neuro stack)
Fast on T4 — should start stepping within a minute of the model downloading.

In [ ]:
import os
os.environ['PYTORCH_CUDA_ALLOC_CONF'] = 'expandable_segments:True'
!python -m src.training.trainer --config configs/train_config.yaml

## 5. Evaluate & inference (run only AFTER training finishes)

In [ ]:
!python -m src.eval.benchmark --config configs/train_config.yaml --checkpoint outputs/run_001/best
!python -m src.inference --config configs/train_config.yaml --checkpoint outputs/run_001/best \
  --lat 35.6895 --lon 139.6917 --question 'Which country is this location in?'

## 6. Save checkpoint to Kaggle output
Files in `/kaggle/working/` persist as notebook output you can download or version.

In [ ]:
!cp -r outputs /kaggle/working/spatial-llm-outputs 2>/dev/null && echo 'Saved to /kaggle/working/' || echo 'Train first'